# Анализ одного запуска

In [ ]:
RUN_ID = "2026-11-02_09-29-16_462268"

In [ ]:
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import yaml

from lle_runs.runs import get_run
from lle_solver.simulations.storage import load_evolution

In [ ]:
run = get_run(RUN_ID)
print(f"Запуск: {run.id}")
print("\nМетаданные")
pprint(run.load_metadata(), sort_dicts=False)

parameters_path = run.path / "parameters.yaml"
if parameters_path.is_file():
    print("\nПараметры")
    pprint(yaml.safe_load(parameters_path.read_text(encoding="utf-8")), sort_dicts=False)

evolution = load_evolution(run.path)
print(f"\nЗаписано кадров: {len(evolution.times)}")

## Эволюция поля

In [ ]:
theta, times, fields = evolution.theta, evolution.times, evolution.values
intensity = evolution.intensity
spacing = theta[1] - theta[0]

### Карта интенсивности

In [ ]:
fig, ax = plt.subplots(layout="constrained")
image = ax.pcolormesh(theta, times, intensity.values, shading="nearest", cmap="magma")
fig.colorbar(image, ax=ax, label=r"$|\psi|^2$")
ax.set(xlabel=r"$\theta$", ylabel=r"$t$", title="Карта интенсивности")
plt.show()

### Интегральная энергия

Интеграл интенсивности по периоду вычисляется как $\Delta\theta \sum_j |\psi_j|^2$ на равномерной периодической сетке.

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax.plot(times, spacing * intensity.values.sum(axis=1), marker="." if times.size == 1 else None)
ax.set(xlabel=r"$t$", ylabel=r"$\int |\psi|^2\,d\theta$", title="Интегральная энергия")
plt.show()

### Профили интенсивности

In [ ]:
PROFILE_TIMES = None

if PROFILE_TIMES is None:
    indices = [0, times.size // 2, times.size - 1]
else:
    indices = [int(np.argmin(np.abs(times - t))) for t in PROFILE_TIMES]
indices = sorted(set(indices))

fig, ax = plt.subplots(layout="constrained")
for index in indices:
    label = f"t = {times[index]:g}"
    ax.plot(theta, intensity[index].values, label=label)
ax.set(xlabel=r"$\theta$", ylabel=r"$|\psi|^2$", title="Профили интенсивности")
if indices:
    ax.legend()
plt.show()

### Спектры

In [ ]:
SPECTRUM_TIMES = None

if SPECTRUM_TIMES is None:
    indices = [0, times.size // 2, times.size - 1]
else:
    indices = [int(np.argmin(np.abs(times - t))) for t in SPECTRUM_TIMES]
indices = sorted(set(indices))

modes = np.fft.fftshift(2 * np.pi * np.fft.fftfreq(theta.size, d=spacing))

fig, ax = plt.subplots(layout="constrained")
for index in indices:
    label = f"t = {times[index]:g}"
    coefficients = np.fft.fftshift(np.fft.fft(fields[index], norm="forward"))
    ax.plot(modes, np.abs(coefficients) ** 2, label=label)
ax.set(xlabel=r"Волновое число $k$", ylabel=r"$|\hat\psi_k|^2$", title="Спектры")
if indices:
    ax.legend()
plt.show()

### Интерактивный просмотр профиля и спектра

In [ ]:
%matplotlib widget

import ipywidgets as widgets
from IPython.display import display


TIME_WINDOW = (50, 160)
MODE_LIMIT = 80
FRAME_STEP = 1

if not np.isfinite(MODE_LIMIT) or MODE_LIMIT <= 0:
    raise ValueError("MODE_LIMIT должен быть положительным конечным числом.")

frame_indices = np.arange(times.size)
if TIME_WINDOW is not None:
    start, end = TIME_WINDOW
    frame_indices = frame_indices[(times >= start) & (times <= end)]
frame_indices = frame_indices[::FRAME_STEP]
if not frame_indices.size:
    raise ValueError("В выбранном интервале нет сохранённых кадров.")

frame_times = times[frame_indices]
frame_intensity = intensity.values[frame_indices]
modes = np.fft.fftshift(2 * np.pi * np.fft.fftfreq(theta.size, d=spacing))
spectra = np.abs(
    np.fft.fftshift(np.fft.fft(fields[frame_indices], axis=1, norm="forward"), axes=1)
) ** 2

spectra[spectra <= 0] = np.nan
positive = spectra[np.isfinite(spectra)]
if positive.size:
    top = 2 * positive.max()
    bottom = max(positive.min() / 2, top * 1e-14)
else:
    bottom, top = 1e-12, 1

profile_ylim = (0, 1.05 * frame_intensity.max() or 1)
spectrum_ylim = (bottom, top)


def show_frames(frame_times, frame_intensity, frame_spectra):
    # ioff предотвращает второй автоматический вывод этой же фигуры.
    with plt.ioff():
        fig, (profile_ax, spectrum_ax) = plt.subplots(1, 2, figsize=(8, 3.6), layout="constrained")
    fig.canvas.header_visible = False
    fig.canvas.toolbar_position = "bottom"
    fig.canvas.layout = widgets.Layout(width="100%")
    profile, = profile_ax.plot(theta, frame_intensity[0])
    spectrum, = spectrum_ax.plot(modes, frame_spectra[0], ".-", linewidth=0.8, markersize=3)
    profile_ax.set(
        xlabel=r"$\theta$", ylabel=r"$|\psi|^2$", title="Профиль интенсивности",
        ylim=profile_ylim,
    )
    spectrum_ax.set(
        xlabel=r"Волновое число $k$", ylabel=r"$|\hat\psi_k|^2$",
        title="Пространственный спектр", yscale="log",
        xlim=(-MODE_LIMIT, MODE_LIMIT), ylim=spectrum_ylim,
    )
    time_label = fig.suptitle(f"t = {frame_times[0]:g}")

    slider = widgets.IntSlider(
        description="Кадр", min=0, max=frame_times.size - 1,
        continuous_update=True, readout=True,
        layout=widgets.Layout(width="100%"),
    )

    play = widgets.Play(
        min=slider.min, max=slider.max, step=1, interval=150,
        description="Воспроизвести", disabled=slider.max == 0,
    )
    play_link = widgets.jslink((play, "value"), (slider, "value"))

    def update_frame(change):
        index = change["new"]
        profile.set_ydata(frame_intensity[index])
        spectrum.set_ydata(frame_spectra[index])
        time_label.set_text(f"t = {frame_times[index]:g}")
        fig.canvas.draw_idle()

    slider.observe(update_frame, names="value")
    return widgets.VBox(
        [widgets.HBox([play, slider]), fig.canvas],
        layout=widgets.Layout(width="100%", max_width="900px"),
    )


display(show_frames(frame_times, frame_intensity, spectra))